## Qwen3-VL-8B Fold 0 분포 보존 증강 LoRA 학습

이 노트북은 팀 공통 `fold_assignments.csv`를 그대로 읽어 Fold 0만 Validation으로 고정합니다. Fold를 다시 계산하지 않으며 Test 데이터에는 접근하지 않습니다.

백본·조건부 프롬프트·7개 LoRA 대상·Fold 0은 이전 실험과 동일하게 유지합니다. Epoch 1은 원본 7,620개를 학습하고, Epoch 2는 원본과 24개 Answer 분포가 정확히 같은 비동일 shuffled copy 7,620개를 학습합니다.

Epoch 1이 끝나면 adapter뿐 아니라 optimizer와 난수 상태도 공통 체크포인트에 저장합니다. 이후 다른 Epoch 2 전략은 이 체크포인트에서 이어서 시작할 수 있습니다. Test 데이터에는 접근하지 않습니다.

### 1. A100, Google Drive, 공통 Fold 파일 확인

학습 전에 GPU와 필수 경로만 검사합니다. 이 셀은 데이터를 학습하거나 모델을 다운로드하지 않습니다. `fold_assignments.csv`는 앞 단계에서 Colab의 scikit-learn 1.6.1로 생성한 파일이어야 합니다.

In [ ]:
from pathlib import Path
import torch
from google.colab import drive

assert torch.cuda.is_available(), 'GPU 런타임이 아닙니다.'
assert torch.cuda.is_bf16_supported(), 'BF16을 지원하는 GPU가 필요합니다.'
drive.mount('/content/drive')

PROJECT_DIR = Path('/content/drive/MyDrive/SNU')
DATA_DIR = PROJECT_DIR / 'data'
TRAIN_CSV = DATA_DIR / 'train.csv'
TRAIN_IMAGE_DIR = DATA_DIR / 'train'
FOLD_CSV = PROJECT_DIR / 'outputs' / 'splits' / 'fold_assignments.csv'

MODEL_NAME = 'Qwen/Qwen3-VL-8B-Instruct'
RUN_NAME = 'fold0_dist_aug_v1'
RUN_DIR = PROJECT_DIR / 'outputs' / 'qwen3_vl_8b_lora' / RUN_NAME
SHARED_EPOCH1_DIR = PROJECT_DIR / 'outputs' / 'qwen3_vl_8b_lora' / 'fold0_shared_epoch1_r16_seed42'
EPOCH1_ADAPTER_DIR = SHARED_EPOCH1_DIR / 'adapter'
EPOCH1_STATE_PATH = SHARED_EPOCH1_DIR / 'training_state.pt'
FINAL_ADAPTER_DIR = RUN_DIR / 'final_adapter'

for required_path in (TRAIN_CSV, TRAIN_IMAGE_DIR, FOLD_CSV):
    assert required_path.exists(), f'필수 경로가 없습니다: {required_path}'
RUN_DIR.mkdir(parents=True, exist_ok=True)
SHARED_EPOCH1_DIR.mkdir(parents=True, exist_ok=True)

print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'프로젝트: {PROJECT_DIR}')
print(f'공통 Fold: {FOLD_CSV}')
print(f'공통 Epoch 1: {SHARED_EPOCH1_DIR}')
print(f'이번 실험 출력: {RUN_DIR}')

### 2. Qwen3-VL LoRA 패키지 설치

Colab의 PyTorch와 CUDA는 교체하지 않고 Transformers, PEFT, bitsandbytes, Qwen 이미지 유틸리티만 설치합니다. 버전을 고정해 다음 실행에서도 같은 API를 사용합니다. 설치 후 import 문제가 발생할 때만 런타임을 재시작하고 1번 셀부터 다시 실행합니다.

In [ ]:
%pip install -q transformers==5.13.0 accelerate==1.14.0 peft==0.19.1 bitsandbytes qwen-vl-utils==0.0.14 ipywidgets
%pip uninstall -y torchao

### 3. 공통 Fold 0과 Train 데이터 무결성 감사

저장된 Fold를 다시 계산하지 않고 `Id`로 Train과 병합합니다. 9,535개 ID가 정확히 한 번씩 존재하고 Fold별 개수가 기존 결과와 같은지 검사합니다. 여기서 통과해야 이후 Validation 점수를 팀원과 비교할 수 있습니다.

In [ ]:
import ast
import hashlib
import json
import math
import random
import re
import shutil
from itertools import permutations

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

SEED = 42
VALIDATION_FOLD = 0
EXPECTED_FOLD_COUNTS = {0: 1915, 1: 1905, 2: 1886, 3: 1911, 4: 1918}
EXPECTED_TRAIN_COUNT = 7620
EXPECTED_VALIDATION_COUNT = 1915

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

train = pd.read_csv(TRAIN_CSV, dtype=str, encoding='utf-8-sig')
folds = pd.read_csv(FOLD_CSV, encoding='utf-8-sig')
assert list(folds.columns) == ['Id', 'fold'], folds.columns.tolist()
assert len(train) == len(folds) == 9535
assert train['Id'].is_unique and folds['Id'].is_unique
assert folds['Id'].notna().all() and folds['fold'].notna().all()
folds['fold'] = folds['fold'].astype(int)
assert folds['fold'].value_counts().sort_index().to_dict() == EXPECTED_FOLD_COUNTS

train = train.merge(folds, on='Id', how='left', validate='one_to_one')
assert train['fold'].notna().all()
train_fold0 = train[train['fold'] != VALIDATION_FOLD].reset_index(drop=True)
validation_fold0 = train[train['fold'] == VALIDATION_FOLD].reset_index(drop=True)
assert len(train_fold0) == EXPECTED_TRAIN_COUNT
assert len(validation_fold0) == EXPECTED_VALIDATION_COUNT
assert set(train_fold0['Id']).isdisjoint(set(validation_fold0['Id']))

def parse_answer(value):
    answer = ast.literal_eval(value) if isinstance(value, str) else list(value)
    assert isinstance(answer, list) and len(answer) == 4
    assert sorted(answer) == [1, 2, 3, 4]
    return [int(number) for number in answer]

assert train['Answer'].apply(parse_answer).map(tuple).nunique() == 24
fold_sha256 = hashlib.sha256(FOLD_CSV.read_bytes()).hexdigest()
print(f'Fold SHA-256: {fold_sha256}')
print(f'Fold 0 Train: {len(train_fold0):,}')
print(f'Fold 0 Validation: {len(validation_fold0):,}')
print('공통 Fold 감사 통과')

### 4. Train 전용 Answer 분포 보존 증강 생성

각 Train ID에 비동일 shuffled copy 한 건을 만듭니다. 다만 임의 순열을 고르는 대신, 증강 후 24개 `Answer` 클래스의 개수가 원본과 정확히 같도록 목표 Answer를 먼저 배정합니다. 이미지 파일은 복사하지 않고 Input 파일명 순서만 바꿉니다.

이 셀은 모든 증강이 실제 비동일 순열인지, 이미지 재배열과 정답 변환이 일치하는지, 원본과 증강의 Answer 분포 차이가 0인지 검사합니다. Validation 1,915개는 변경하지 않습니다.

In [ ]:
IDENTITY_ORDER = (1, 2, 3, 4)
source_answers = [tuple(parse_answer(value)) for value in train_fold0['Answer']]
sorted_indices = sorted(
    range(len(train_fold0)),
    key=lambda index: (source_answers[index], train_fold0.iloc[index]['Id']),
)
sorted_answers = [source_answers[index] for index in sorted_indices]
max_class_count = max(pd.Series(source_answers).value_counts())
distribution_shift = int(max_class_count)
shifted_answers = sorted_answers[distribution_shift:] + sorted_answers[:distribution_shift]
assert all(source != target for source, target in zip(sorted_answers, shifted_answers))

target_answer_by_index = [None] * len(train_fold0)
for sorted_position, row_index in enumerate(sorted_indices):
    target_answer_by_index[row_index] = shifted_answers[sorted_position]
assert all(target is not None for target in target_answer_by_index)

def augment_row(row, target_answer):
    result = row.copy()
    old_inputs = [row[f'Input_{index}'] for index in IDENTITY_ORDER]
    old_answer = parse_answer(row['Answer'])
    target_answer = list(target_answer)
    order = tuple(old_answer.index(position) + 1 for position in target_answer)
    assert order != IDENTITY_ORDER
    for new_index, old_index in enumerate(order, start=1):
        result[f'Input_{new_index}'] = old_inputs[old_index - 1]
    new_answer = [old_answer[old_index - 1] for old_index in order]
    assert new_answer == target_answer
    result['Answer'] = str(new_answer)
    result['_variant'] = 'distribution_preserving_augmented'
    result['_input_order'] = str(list(order))
    return result

original_train = train_fold0.copy()
original_train['_variant'] = 'original'
original_train['_input_order'] = str(list(IDENTITY_ORDER))
augmented_train = pd.DataFrame([
    augment_row(row, target_answer_by_index[index])
    for index, (_, row) in enumerate(train_fold0.iterrows())
])
train_examples = pd.concat([original_train, augmented_train], ignore_index=True)
validation_examples = validation_fold0.copy()
validation_examples['_variant'] = 'original'
validation_examples['_input_order'] = str(list(IDENTITY_ORDER))

assert len(train_examples) == 15240
assert train_examples['_variant'].value_counts().to_dict() == {
    'original': 7620, 'distribution_preserving_augmented': 7620
}
assert len(validation_examples) == 1915
assert (validation_examples['_variant'] == 'original').all()
original_distribution = original_train['Answer'].apply(parse_answer).map(tuple).value_counts().sort_index()
augmented_distribution = augmented_train['Answer'].apply(parse_answer).map(tuple).value_counts().sort_index()
assert original_distribution.equals(augmented_distribution)
assert augmented_distribution.size == 24
assert all(order != str(list(IDENTITY_ORDER)) for order in augmented_train['_input_order'])

audit_index = 0
print('원본 Input:', [original_train.iloc[audit_index][f'Input_{i}'] for i in IDENTITY_ORDER])
print('원본 Answer:', original_train.iloc[audit_index]['Answer'])
print('증강 Input 순서:', augmented_train.iloc[audit_index]['_input_order'])
print('증강 Input:', [augmented_train.iloc[audit_index][f'Input_{i}'] for i in IDENTITY_ORDER])
print('증강 Answer:', augmented_train.iloc[audit_index]['Answer'])
print(f'Answer classes: {augmented_distribution.size}')
print(f'원본-증강 클래스별 최대 개수 차이: {(original_distribution - augmented_distribution).abs().max()}')
print(f'분포 보존 순환 이동 크기: {distribution_shift:,}')
print(f'Train examples: {len(train_examples):,}')
print(f'Validation examples: {len(validation_examples):,}')

### 5. 조건부 프롬프트와 chronological target 정의

Caption의 시간 접속사 수에 따라 단계가 뚜렷한 동작과 연속 동작에 서로 다른 관찰 힌트를 줍니다. 모델 출력은 공식 Answer가 아니라 가장 이른 이미지부터 나열한 `chronological order`입니다. 학습과 추론에서 같은 의미를 사용하고, 평가 직전에 공식 Answer 형식으로 역변환합니다.

In [ ]:
CLAUSE_PATTERN = re.compile(
    r'\b(then|after which|after that|and finally|finally|next|before that|followed by|first|before|after)\b,?',
    flags=re.IGNORECASE,
)
STRUCTURED_HINT = (
    'This caption describes distinct sequential stages, often marked by words like '
    "'then', 'after which', or 'finally'. Match each stage of the story to its corresponding image."
)
CONTINUOUS_HINT = (
    'This caption describes a single continuous action or a brief moment. '
    'Focus on subtle visual cues such as camera movement direction, object position changes, '
    'hand/body motion, and lighting changes to infer the temporal order.'
)

def split_clauses(sentence, max_clauses=4):
    sentence = sentence.strip()
    parts = CLAUSE_PATTERN.split(sentence)
    clauses = []
    current = parts[0].strip(' ,;.')
    for index in range(1, len(parts), 2):
        if current:
            clauses.append(current)
        current = parts[index + 1].strip(' ,;.')
    if current:
        clauses.append(current)
    clauses = [clause for clause in clauses if clause]
    if len(clauses) > max_clauses:
        clauses = clauses[:max_clauses - 1] + [' '.join(clauses[max_clauses - 1:])]
    return clauses or [sentence]

def answer_to_chronological_order(answer):
    answer = parse_answer(answer)
    return [index for _, index in sorted((position, index) for index, position in enumerate(answer, 1))]

def chronological_order_to_answer(order):
    assert isinstance(order, list) and sorted(order) == [1, 2, 3, 4]
    answer = [0, 0, 0, 0]
    for position, image_number in enumerate(order, start=1):
        answer[image_number - 1] = position
    return answer

def build_prompt_messages(row, image_dir):
    sentence = row['Sentence']
    hint = STRUCTURED_HINT if len(split_clauses(sentence)) >= 3 else CONTINUOUS_HINT
    content = []
    for image_number in IDENTITY_ORDER:
        image_path = image_dir / row['Id'] / row[f'Input_{image_number}']
        assert image_path.is_file(), image_path
        content.append({'type': 'image', 'image': str(image_path)})
        content.append({'type': 'text', 'text': f'\nImage {image_number}\n'})
    content.append({
        'type': 'text',
        'text': (
            f'Thinking about the sentence: \"{sentence}\"\n'
            f'{hint}\n'
            'Look at the 4 images above labeled Image 1 to Image 4. '
            'Determine the correct chronological order of these images to match the sentence. '
            'Provide the answer ONLY as a Python list of integers. '
            'Example: [1, 2, 3, 4]'
        ),
    })
    return [{'role': 'user', 'content': content}]

def build_training_messages(row, image_dir):
    messages = build_prompt_messages(row, image_dir)
    target = answer_to_chronological_order(row['Answer'])
    messages.append({
        'role': 'assistant',
        'content': [{'type': 'text', 'text': str(target)}],
    })
    return messages

for answer in permutations(IDENTITY_ORDER):
    answer = list(answer)
    assert chronological_order_to_answer(answer_to_chronological_order(answer)) == answer
print('24개 Answer ↔ chronological order 왕복 변환 통과')

### 6. Dataset과 completion-only loss 정의

모델에는 네 이미지, Caption, 프롬프트가 모두 입력되지만 loss는 assistant가 출력해야 하는 네 숫자 토큰에만 적용합니다. 프롬프트 토큰까지 정답으로 학습하는 구현 오류를 막기 위해 첫 배치에서 마스킹을 직접 감사합니다.

In [ ]:
from dataclasses import dataclass
from torch.utils.data import DataLoader, Dataset
from qwen_vl_utils import process_vision_info

class OrderingDataset(Dataset):
    def __init__(self, dataframe, image_dir, training):
        self.dataframe = dataframe.reset_index(drop=True)
        self.image_dir = image_dir
        self.training = training

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):
        row = self.dataframe.iloc[index]
        messages = (
            build_training_messages(row, self.image_dir)
            if self.training
            else build_prompt_messages(row, self.image_dir)
        )
        return {'Id': row['Id'], 'messages': messages, 'Answer': row['Answer']}

@dataclass
class CompletionOnlyCollator:
    processor: object

    def __call__(self, examples):
        assert len(examples) == 1, '학습 batch size는 1로 고정합니다.'
        messages = examples[0]['messages']
        assert len(messages) == 2
        full_text = self.processor.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=False
        )
        prompt_text = self.processor.apply_chat_template(
            messages[:-1], tokenize=False, add_generation_prompt=True
        )
        image_inputs, video_inputs = process_vision_info(messages)
        assert image_inputs is not None and len(image_inputs) == 4
        full = self.processor(
            text=[full_text], images=image_inputs, videos=video_inputs,
            padding=True, return_tensors='pt',
        )
        prompt = self.processor(
            text=[prompt_text], images=image_inputs, videos=video_inputs,
            padding=True, return_tensors='pt',
        )
        prompt_length = int(prompt['attention_mask'][0].sum().item())
        full_length = int(full['attention_mask'][0].sum().item())
        assert prompt_length < full_length
        assert torch.equal(
            full['input_ids'][0, :prompt_length],
            prompt['input_ids'][0, :prompt_length],
        ), 'Prompt가 전체 학습 문장의 prefix가 아닙니다.'
        labels = full['input_ids'].clone()
        labels[full['attention_mask'] == 0] = -100
        labels[:, :prompt_length] = -100
        assert int((labels != -100).sum().item()) > 0
        full['labels'] = labels
        return dict(full)

original_train_dataset = OrderingDataset(original_train, TRAIN_IMAGE_DIR, training=True)
augmented_train_dataset = OrderingDataset(augmented_train, TRAIN_IMAGE_DIR, training=True)
validation_dataset = OrderingDataset(validation_examples, TRAIN_IMAGE_DIR, training=False)
print(f'Epoch 1 Original Dataset: {len(original_train_dataset):,}')
print(f'Epoch 2 Augmented Dataset: {len(augmented_train_dataset):,}')
print(f'Validation Dataset: {len(validation_dataset):,}')

### 7. Qwen3-VL-8B와 7개 LoRA 모듈 로드

기본 모델 전체를 업데이트하지 않고 attention의 `q/k/v/o_proj`와 MLP의 `gate/up/down_proj`에만 LoRA를 연결합니다. `r=16`, `alpha=32`, `dropout=0.05`를 사용합니다. 모델 구조에서 7개 모듈이 실제로 모두 발견되는지도 검사합니다.

In [ ]:
import transformers
import peft
from transformers import AutoModelForImageTextToText, AutoProcessor
from peft import LoraConfig, PeftModel, TaskType, get_peft_model

LORA_TARGET_MODULES = [
    'q_proj', 'k_proj', 'v_proj', 'o_proj',
    'gate_proj', 'up_proj', 'down_proj',
]
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05

print(f'transformers: {transformers.__version__}')
print(f'peft: {peft.__version__}')
processor = AutoProcessor.from_pretrained(MODEL_NAME)
processor.tokenizer.padding_side = 'right'

base_model = AutoModelForImageTextToText.from_pretrained(
    MODEL_NAME,
    dtype=torch.bfloat16,
    device_map='auto',
    low_cpu_mem_usage=True,
)
available_suffixes = {name.rsplit('.', 1)[-1] for name, _ in base_model.named_modules()}
missing_targets = sorted(set(LORA_TARGET_MODULES) - available_suffixes)
assert not missing_targets, f'모델에서 LoRA 대상 모듈을 찾지 못했습니다: {missing_targets}'

if FINAL_ADAPTER_DIR.is_dir():
    model = PeftModel.from_pretrained(base_model, FINAL_ADAPTER_DIR, is_trainable=False)
    TRAINING_REQUIRED = False
    START_PHASE = 2
    print('저장된 final_adapter를 불러왔습니다. 학습 셀은 건너뜁니다.')
elif EPOCH1_ADAPTER_DIR.is_dir() and EPOCH1_STATE_PATH.is_file():
    model = PeftModel.from_pretrained(base_model, EPOCH1_ADAPTER_DIR, is_trainable=True)
    TRAINING_REQUIRED = True
    START_PHASE = 1
    print('공통 Epoch 1 adapter와 학습 상태를 불러왔습니다. Epoch 2부터 계속합니다.')
elif EPOCH1_ADAPTER_DIR.exists() or EPOCH1_STATE_PATH.exists():
    raise RuntimeError(
        '공통 Epoch 1 adapter와 training_state.pt가 모두 있어야 합니다. '
        f'불완전한 폴더를 확인하세요: {SHARED_EPOCH1_DIR}'
    )
else:
    lora_config = LoraConfig(
        task_type=TaskType.CAUSAL_LM,
        r=LORA_R,
        lora_alpha=LORA_ALPHA,
        lora_dropout=LORA_DROPOUT,
        bias='none',
        target_modules=LORA_TARGET_MODULES,
    )
    model = get_peft_model(base_model, lora_config)
    TRAINING_REQUIRED = True
    START_PHASE = 0

model.config.use_cache = False
if TRAINING_REQUIRED:
    model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={'use_reentrant': False})
    model.enable_input_require_grads()
model.print_trainable_parameters()
print('LoRA 대상:', LORA_TARGET_MODULES)

### 8. 첫 배치 loss 마스킹 감사

실제 모델 입력 한 건을 만들고 assistant 정답 토큰만 label로 남았는지 확인합니다. 이 셀은 forward/backward 학습을 수행하지 않으므로 빠르게 끝나야 합니다.

In [ ]:
collator = CompletionOnlyCollator(processor)
audit_batch = collator([original_train_dataset[0]])
supervised_tokens = int((audit_batch['labels'] != -100).sum().item())
non_padding_tokens = int(audit_batch['attention_mask'].sum().item())
assert 0 < supervised_tokens < non_padding_tokens
print(f'Assistant supervised tokens: {supervised_tokens}')
print(f'전체 non-padding tokens: {non_padding_tokens}')
print('Completion-only loss 감사 통과')
del audit_batch
torch.cuda.empty_cache()

### 9. Fold 0 LoRA 학습

직접 학습 반복문으로 Epoch 1은 원본 7,620개, Epoch 2는 분포 보존 shuffled copy 7,620개를 학습합니다. batch size 1, gradient accumulation 8, learning rate `2e-4`를 이전 실험과 동일하게 유지합니다.

Epoch 1 뒤에는 공통 adapter와 `training_state.pt`에 optimizer·난수 상태를 함께 저장합니다. 그래서 다음 실험은 Epoch 1을 다시 학습하지 않고 같은 상태에서 다른 Epoch 2 전략을 비교할 수 있습니다. 대회 Exact Match는 다음 Validation 생성 셀에서 별도로 측정합니다.

In [ ]:
import time
import bitsandbytes as bnb

BATCH_SIZE = 1
GRADIENT_ACCUMULATION_STEPS = 8
LEARNING_RATE = 2e-4
WEIGHT_DECAY = 0.0
DATALOADER_NUM_WORKERS = 2
DATALOADER_PREFETCH_FACTOR = 2
LOG_EVERY_OPTIMIZER_STEPS = 10

training_phases = [
    {
        'name': 'epoch1_original',
        'dataset': original_train_dataset,
        'adapter_dir': EPOCH1_ADAPTER_DIR,
    },
    {
        'name': 'epoch2_distribution_preserving_augmented',
        'dataset': augmented_train_dataset,
        'adapter_dir': FINAL_ADAPTER_DIR,
    },
]
assert [len(phase['dataset']) for phase in training_phases] == [7620, 7620]

run_config = {
    'scope': 'train_fold0_only_no_test_access',
    'fold_sha256': fold_sha256,
    'validation_fold': VALIDATION_FOLD,
    'unique_train_samples': len(train_fold0),
    'training_examples': len(train_examples),
    'validation_samples': len(validation_examples),
    'training_engine': 'manual_pytorch_loop',
    'phase_order': ['epoch1_original', 'epoch2_distribution_preserving_augmented'],
    'augmentation': 'exact_24_class_distribution_preserving_non_identity_permutation',
    'augmentation_distribution_shift': distribution_shift,
    'resume_scope': 'epoch_boundary_with_optimizer_and_rng_state',
    'model_name': MODEL_NAME,
    'learning_rate': LEARNING_RATE,
    'learning_rate_schedule': 'constant',
    'effective_batch_size': BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS,
    'dataloader': {
        'num_workers': DATALOADER_NUM_WORKERS,
        'prefetch_factor': DATALOADER_PREFETCH_FACTOR,
        'persistent_workers': True,
    },
    'lora': {
        'r': LORA_R, 'alpha': LORA_ALPHA, 'dropout': LORA_DROPOUT,
        'target_modules': LORA_TARGET_MODULES,
    },
}
with (RUN_DIR / 'run_config.json').open('w', encoding='utf-8') as file:
    json.dump(run_config, file, ensure_ascii=False, indent=2)

summary_path = RUN_DIR / 'training_summary.json'
log_path = RUN_DIR / 'training_log.csv'
shared_summary_path = SHARED_EPOCH1_DIR / 'training_summary.json'
shared_log_path = SHARED_EPOCH1_DIR / 'training_log.csv'
phase_summaries = []
log_records = []
resume_summary_path = summary_path if summary_path.is_file() else shared_summary_path
resume_log_path = log_path if log_path.is_file() else shared_log_path
if START_PHASE > 0 and resume_summary_path.is_file():
    with resume_summary_path.open(encoding='utf-8') as file:
        phase_summaries = json.load(file).get('phases', [])
if START_PHASE > 0 and resume_log_path.is_file():
    log_records = pd.read_csv(resume_log_path).to_dict('records')

if TRAINING_REQUIRED:
    torch.backends.cuda.matmul.allow_tf32 = True
    trainable_parameters = [parameter for parameter in model.parameters() if parameter.requires_grad]
    assert trainable_parameters, '학습 가능한 LoRA 파라미터가 없습니다.'
    training_device = trainable_parameters[0].device
    optimizer = bnb.optim.AdamW8bit(
        trainable_parameters, lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY
    )
    optimizer.zero_grad(set_to_none=True)
    global_optimizer_step = 0
    if START_PHASE == 1:
        saved_state = torch.load(EPOCH1_STATE_PATH, map_location='cpu', weights_only=False)
        assert saved_state['fold_sha256'] == fold_sha256
        assert saved_state['lora_target_modules'] == LORA_TARGET_MODULES
        assert saved_state['lora_r'] == LORA_R
        optimizer.load_state_dict(saved_state['optimizer'])
        for optimizer_state in optimizer.state.values():
            for key, value in optimizer_state.items():
                if torch.is_tensor(value):
                    optimizer_state[key] = value.to(training_device)
        global_optimizer_step = int(saved_state['global_optimizer_step'])
        random.setstate(saved_state['python_rng_state'])
        np.random.set_state(saved_state['numpy_rng_state'])
        torch.set_rng_state(saved_state['torch_rng_state'])
        torch.cuda.set_rng_state_all(saved_state['cuda_rng_state_all'])
        print(f'Optimizer/RNG 상태 복원 완료: step {global_optimizer_step:,}')

    for phase_index in range(START_PHASE, len(training_phases)):
        phase = training_phases[phase_index]
        generator = torch.Generator()
        generator.manual_seed(SEED + phase_index)
        loader = DataLoader(
            phase['dataset'],
            batch_size=BATCH_SIZE,
            shuffle=True,
            collate_fn=collator,
            num_workers=DATALOADER_NUM_WORKERS,
            prefetch_factor=DATALOADER_PREFETCH_FACTOR,
            persistent_workers=True,
            pin_memory=True,
            generator=generator,
        )
        model.train()
        model.config.use_cache = False
        phase_loss_sum = 0.0
        phase_started = time.perf_counter()
        progress = tqdm(loader, desc=phase['name'])

        for micro_step, batch in enumerate(progress, start=1):
            batch = {
                key: value.to(training_device, non_blocking=True)
                if torch.is_tensor(value) else value
                for key, value in batch.items()
            }
            with torch.autocast(device_type='cuda', dtype=torch.bfloat16):
                outputs = model(**batch)
                raw_loss = outputs.loss
                loss = raw_loss / GRADIENT_ACCUMULATION_STEPS
            assert torch.isfinite(raw_loss), f'비정상 loss: {raw_loss.item()}'
            loss.backward()
            phase_loss_sum += float(raw_loss.detach().item())

            should_update = (
                micro_step % GRADIENT_ACCUMULATION_STEPS == 0
                or micro_step == len(loader)
            )
            if should_update:
                grad_norm = torch.nn.utils.clip_grad_norm_(trainable_parameters, max_norm=1.0)
                optimizer.step()
                optimizer.zero_grad(set_to_none=True)
                global_optimizer_step += 1
                log_records.append({
                    'phase': phase['name'],
                    'micro_step': micro_step,
                    'optimizer_step': global_optimizer_step,
                    'loss': float(raw_loss.detach().item()),
                    'grad_norm': float(grad_norm),
                    'learning_rate': LEARNING_RATE,
                })
                if global_optimizer_step == 1 or global_optimizer_step % LOG_EVERY_OPTIMIZER_STEPS == 0:
                    progress.set_postfix(
                        loss=f'{raw_loss.item():.4f}',
                        optimizer_step=global_optimizer_step,
                    )

        elapsed_seconds = time.perf_counter() - phase_started
        phase['adapter_dir'].mkdir(parents=True, exist_ok=True)
        model.save_pretrained(phase['adapter_dir'], safe_serialization=True)
        processor.save_pretrained(phase['adapter_dir'])
        phase_summaries = [
            summary for summary in phase_summaries if summary.get('name') != phase['name']
        ]
        phase_summaries.append({
            'name': phase['name'],
            'sample_count': len(phase['dataset']),
            'mean_training_loss': phase_loss_sum / len(phase['dataset']),
            'elapsed_seconds': elapsed_seconds,
            'adapter_dir': str(phase['adapter_dir']),
        })
        training_summary = {
            'training_engine': 'manual_pytorch_loop',
            'phases': phase_summaries,
            'final_adapter_dir': str(FINAL_ADAPTER_DIR),
            'exact_match_evaluated': False,
        }
        if phase_index == 0:
            shared_summary = {
                'training_engine': 'manual_pytorch_loop',
                'phases': phase_summaries,
                'shared_epoch1_adapter_dir': str(EPOCH1_ADAPTER_DIR),
                'optimizer_and_rng_state': str(EPOCH1_STATE_PATH),
                'exact_match_evaluated': False,
            }
            with shared_summary_path.open('w', encoding='utf-8') as file:
                json.dump(shared_summary, file, ensure_ascii=False, indent=2)
            pd.DataFrame(log_records).to_csv(
                shared_log_path, index=False, encoding='utf-8-sig'
            )
            torch.save({
                'completed_phase': 'epoch1_original',
                'fold_sha256': fold_sha256,
                'lora_target_modules': LORA_TARGET_MODULES,
                'lora_r': LORA_R,
                'optimizer': optimizer.state_dict(),
                'global_optimizer_step': global_optimizer_step,
                'python_rng_state': random.getstate(),
                'numpy_rng_state': np.random.get_state(),
                'torch_rng_state': torch.get_rng_state(),
                'cuda_rng_state_all': torch.cuda.get_rng_state_all(),
            }, EPOCH1_STATE_PATH)
            print(f'공통 Epoch 1 학습 상태 저장: {EPOCH1_STATE_PATH}')
        with summary_path.open('w', encoding='utf-8') as file:
            json.dump(training_summary, file, ensure_ascii=False, indent=2)
        pd.DataFrame(log_records).to_csv(log_path, index=False, encoding='utf-8-sig')
        print(f"{phase['name']} 완료 및 저장: {phase['adapter_dir']}")

    model.gradient_checkpointing_disable()
    model.config.use_cache = True
    print(f'전체 학습 완료: {FINAL_ADAPTER_DIR}')
else:
    print(f'기존 학습 완료 adapter 사용: {FINAL_ADAPTER_DIR}')

### 10. 전체 Fold 0 Validation 생성 평가

분포 보존 증강까지 학습한 `final_adapter/`를 사용해 Fold 0의 1,915개 원본 샘플에 greedy generation을 수행합니다. 가장 중요한 지표는 공식 Answer 네 자리가 모두 맞은 비율인 `Exact Match Accuracy`입니다. 목표는 현재 공통 기준선 약 0.54를 넘는 것입니다.

In [ ]:
def parse_chronological_output(text):
    start = text.find('[')
    end = text.rfind(']')
    if start == -1 or end == -1 or start >= end:
        return [1, 2, 3, 4], False, 'missing_brackets'
    try:
        value = ast.literal_eval(text[start:end + 1])
    except (SyntaxError, ValueError):
        return [1, 2, 3, 4], False, 'invalid_list'
    if not isinstance(value, list) or len(value) != 4 or sorted(value) != [1, 2, 3, 4]:
        return [1, 2, 3, 4], False, 'not_permutation'
    return chronological_order_to_answer([int(number) for number in value]), True, None

def compute_ordering_metrics(targets, predictions):
    targets = np.asarray(targets, dtype=int)
    predictions = np.asarray(predictions, dtype=int)
    exact = float(np.mean(np.all(targets == predictions, axis=1)))
    position = float(np.mean(targets == predictions))
    pairwise_values = []
    for target, prediction in zip(targets, predictions):
        correct = 0
        total = 0
        for first in range(4):
            for second in range(first + 1, 4):
                correct += (prediction[first] < prediction[second]) == (target[first] < target[second])
                total += 1
        pairwise_values.append(correct / total)
    pairwise = float(np.mean(pairwise_values))
    return {
        'exact_match_accuracy': exact,
        'position_accuracy': position,
        'pairwise_ordering_accuracy': pairwise,
        'mean_kendall_tau': 2.0 * pairwise - 1.0,
    }

model.config.use_cache = True
model.eval()
processor.tokenizer.padding_side = 'left'
EVAL_BATCH_SIZE = 4
results = []

for start_index in tqdm(range(0, len(validation_examples), EVAL_BATCH_SIZE), desc='Fold 0 Validation'):
    batch_rows = validation_examples.iloc[start_index:start_index + EVAL_BATCH_SIZE]
    batch_messages = [
        build_prompt_messages(row, TRAIN_IMAGE_DIR)
        for _, row in batch_rows.iterrows()
    ]
    batch_texts = [
        processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        for messages in batch_messages
    ]
    all_images = []
    for messages in batch_messages:
        image_inputs, video_inputs = process_vision_info(messages)
        assert not video_inputs
        all_images.extend(image_inputs)
    inputs = processor(
        text=batch_texts, images=all_images, padding=True, return_tensors='pt'
    ).to(model.device)
    with torch.inference_mode():
        generated = model.generate(
            **inputs, max_new_tokens=16, do_sample=False, num_beams=1
        )
    generated_only = generated[:, inputs['input_ids'].shape[1]:]
    output_texts = processor.batch_decode(generated_only, skip_special_tokens=True)
    for (_, row), raw_output in zip(batch_rows.iterrows(), output_texts):
        prediction, parse_ok, parse_error = parse_chronological_output(raw_output)
        results.append({
            'Id': row['Id'],
            'target_answer': str(parse_answer(row['Answer'])),
            'predicted_answer': str(prediction),
            'parse_success': parse_ok,
            'parse_error': parse_error,
            'raw_output': raw_output,
        })

assert len(results) == EXPECTED_VALIDATION_COUNT
targets = [ast.literal_eval(row['target_answer']) for row in results]
predictions = [ast.literal_eval(row['predicted_answer']) for row in results]
metrics = compute_ordering_metrics(targets, predictions)
parse_success_rate = float(np.mean([row['parse_success'] for row in results]))
identity_mask = np.array([target == [1, 2, 3, 4] for target in targets])
shuffled_exact = float(np.mean(
    np.all(np.asarray(targets)[~identity_mask] == np.asarray(predictions)[~identity_mask], axis=1)
))

validation_predictions_path = RUN_DIR / 'fold0_validation_predictions.csv'
validation_summary_path = RUN_DIR / 'fold0_validation_summary.json'
pd.DataFrame(results).to_csv(validation_predictions_path, index=False, encoding='utf-8-sig')
validation_summary = {
    'scope': 'train_fold0_validation_only_no_test_access',
    'sample_count': len(results),
    'fold_sha256': fold_sha256,
    'metrics': metrics,
    'shuffled_only_exact_match_accuracy': shuffled_exact,
    'parse_success_rate': parse_success_rate,
    'adapter': str(FINAL_ADAPTER_DIR),
}
with validation_summary_path.open('w', encoding='utf-8') as file:
    json.dump(validation_summary, file, ensure_ascii=False, indent=2)

print(json.dumps(validation_summary, ensure_ascii=False, indent=2))
print(f'예측 저장: {validation_predictions_path}')
print(f'요약 저장: {validation_summary_path}')

### 11. 공통 Epoch 1 adapter 비교 평가

재학습 없이 공통 Epoch 1 adapter를 같은 Fold 0의 1,915개로 평가합니다. 분포 보존 Epoch 2가 실제로 개선됐는지 동일한 모델·프롬프트·생성 설정으로 직접 비교합니다. 결과는 이번 실험 폴더의 `epoch1_validation_*` 파일에 따로 저장합니다.

In [ ]:
assert EPOCH1_ADAPTER_DIR.is_dir(), f'Epoch 1 adapter가 없습니다: {EPOCH1_ADAPTER_DIR}'
assert 'default' in model.peft_config, model.peft_config.keys()
assert validation_summary['adapter'] == str(FINAL_ADAPTER_DIR)

EPOCH1_ADAPTER_NAME = 'epoch1_compare'
if EPOCH1_ADAPTER_NAME not in model.peft_config:
    model.load_adapter(
        str(EPOCH1_ADAPTER_DIR),
        adapter_name=EPOCH1_ADAPTER_NAME,
        is_trainable=False,
    )
model.set_adapter(EPOCH1_ADAPTER_NAME, inference_mode=True)
model.config.use_cache = True
model.eval()
processor.tokenizer.padding_side = 'left'
epoch1_results = []

for start_index in tqdm(
    range(0, len(validation_examples), EVAL_BATCH_SIZE),
    desc='Epoch 1 Fold 0 Validation',
):
    batch_rows = validation_examples.iloc[start_index:start_index + EVAL_BATCH_SIZE]
    batch_messages = [
        build_prompt_messages(row, TRAIN_IMAGE_DIR)
        for _, row in batch_rows.iterrows()
    ]
    batch_texts = [
        processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        for messages in batch_messages
    ]
    all_images = []
    for messages in batch_messages:
        image_inputs, video_inputs = process_vision_info(messages)
        assert not video_inputs
        all_images.extend(image_inputs)
    inputs = processor(
        text=batch_texts, images=all_images, padding=True, return_tensors='pt'
    ).to(model.device)
    with torch.inference_mode():
        generated = model.generate(
            **inputs, max_new_tokens=16, do_sample=False, num_beams=1
        )
    generated_only = generated[:, inputs['input_ids'].shape[1]:]
    output_texts = processor.batch_decode(generated_only, skip_special_tokens=True)
    for (_, row), raw_output in zip(batch_rows.iterrows(), output_texts):
        prediction, parse_ok, parse_error = parse_chronological_output(raw_output)
        epoch1_results.append({
            'Id': row['Id'],
            'target_answer': str(parse_answer(row['Answer'])),
            'predicted_answer': str(prediction),
            'parse_success': parse_ok,
            'parse_error': parse_error,
            'raw_output': raw_output,
        })

assert len(epoch1_results) == EXPECTED_VALIDATION_COUNT
epoch1_targets = [ast.literal_eval(row['target_answer']) for row in epoch1_results]
epoch1_predictions = [ast.literal_eval(row['predicted_answer']) for row in epoch1_results]
epoch1_metrics = compute_ordering_metrics(epoch1_targets, epoch1_predictions)
epoch1_parse_success_rate = float(np.mean([row['parse_success'] for row in epoch1_results]))
epoch1_identity_mask = np.array([target == [1, 2, 3, 4] for target in epoch1_targets])
epoch1_shuffled_exact = float(np.mean(
    np.all(
        np.asarray(epoch1_targets)[~epoch1_identity_mask]
        == np.asarray(epoch1_predictions)[~epoch1_identity_mask],
        axis=1,
    )
))
final_exact_match = validation_summary['metrics']['exact_match_accuracy']
epoch1_exact_match = epoch1_metrics['exact_match_accuracy']

epoch1_predictions_path = RUN_DIR / 'epoch1_validation_predictions.csv'
epoch1_summary_path = RUN_DIR / 'epoch1_validation_summary.json'
pd.DataFrame(epoch1_results).to_csv(
    epoch1_predictions_path, index=False, encoding='utf-8-sig'
)
epoch1_summary = {
    'scope': 'train_fold0_validation_only_no_test_access',
    'sample_count': len(epoch1_results),
    'fold_sha256': fold_sha256,
    'metrics': epoch1_metrics,
    'shuffled_only_exact_match_accuracy': epoch1_shuffled_exact,
    'parse_success_rate': epoch1_parse_success_rate,
    'adapter': str(EPOCH1_ADAPTER_DIR),
    'final_adapter_exact_match_accuracy': final_exact_match,
    'exact_match_difference_vs_final': epoch1_exact_match - final_exact_match,
}
with epoch1_summary_path.open('w', encoding='utf-8') as file:
    json.dump(epoch1_summary, file, ensure_ascii=False, indent=2)

print(json.dumps(epoch1_summary, ensure_ascii=False, indent=2))
print(f'예측 저장: {epoch1_predictions_path}')
print(f'요약 저장: {epoch1_summary_path}')
model.set_adapter('default', inference_mode=True)
print('비교 평가 완료 후 final_adapter(default)를 다시 활성화했습니다.')